In [1]:
import os
import pandas as pd
import re

# Path to your Raw Data folder
base_path = r"E:\Cheeseboard-rawdata-editted_Stella_edit"

# Regex to capture: YYYY-MM-DD_{Animal}_trial_{n}.avi
pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)\.avi")

records = []

# Walk through Raw Data folder
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".avi"):
            m = pattern.match(file)
            if m:
                date_str, animal_name, trial_num = m.groups()
                date = date_str.replace("-", "")  # YYYYMMDD
                # if date == '20250607' and animal_name == 'AD':
                #     print(f"Found: {file}")
                trial_num = int(trial_num)
                records.append({
                    "id": animal_name,
                    "date": date,
                    "trial": trial_num
                })
            else:
                print(f"⚠️ Filename format unexpected: {file}")

# Make DataFrame
df_trials = pd.DataFrame(records)

# Group by id/date to compute number of trials
def check_trials(trials, animal_name, date):
    trials_sorted = sorted(trials)
    n_trials = len(trials_sorted)
    max_trial = max(trials_sorted)
    if trials_sorted != list(range(1, max_trial + 1)):
        print(f"⚠️ {animal_name}-{date}: Missing trial(s) in sequence: {trials_sorted}")
    return n_trials

trial_info = (
    df_trials.groupby(["id", "date"])["trial"]
    .apply(list)
    .reset_index()
)

trial_info["n_trials"] = trial_info.apply(
    lambda row: check_trials(row["trial"], row["id"], row["date"]), axis=1
)

# Merge into final df (one row per id/date)
df = trial_info.drop(columns="trial").sort_values(by=["id", "date"]).reset_index(drop=True)

print(df.head())

⚠️ Filename format unexpected: ._2025-10-03_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-03_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-10-04_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-10-04_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-05_AN_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-05_AM_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-05_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-02_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-10-02_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-06-15_AD_trial_1.avi
⚠️ Filename format unexpected: ._2025-09-30_AN_trial_8.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_3.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-09-30_AM_trial_8.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_4.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_5.a

In [19]:
# Save df as csv
df.to_csv("sessions_metadata.csv", index=False)

`trial_info` is a dataframe with columns:
* id (string)
* date (in YYYYMMDD format)
* trial (python list): for example, [1,2,3,7,8]


Below is a script to make a dataframe, with columns:
* id
* date
* trial_id
* trial_type
Where id and date stay the same as before, but instead of one session per row, make it one trial per row. Trial type column can be empty.

In [13]:
import pandas as pd

# --- Expand each list into separate rows ---
df_expanded = trial_info.explode("trial").reset_index(drop=True)

# --- Rename columns ---
df_expanded = df_expanded.rename(columns={"trial": "trial_id"})

# --- Add an empty 'trial_type' column ---
df_expanded["trial_type"] = None

# Order rows by trial_id
df_expanded = df_expanded.sort_values(by=["id", "date", "trial_id"]).reset_index(drop=True)


In [ ]:
# Save df_expanded as csv
df_expanded.to_csv("trials_metadata.csv", index=False)

In [15]:
# Get the unique animal IDs
unique_animals = df["id"].unique()
print("Unique animal IDs:", unique_animals)

Unique animal IDs: ['A' 'AA' 'AB' 'AC' 'AD' 'AE' 'AF' 'AG' 'AH' 'AI' 'AJ' 'AL' 'AM' 'AN' 'AO'
 'AP' 'AQ' 'AR' 'AS' 'AT' 'AU' 'AV' 'B' 'C' 'D' 'E' 'EE' 'EF' 'F' 'G' 'H'
 'HA' 'HB' 'I' 'J' 'K' 'L' 'M' 'MA' 'MB' 'N' 'O' 'P' 'Q' 'R' 'S' 'T']


In [16]:
# Create a df with columns: id, genotype, with the genotype column empty
df_genotype = pd.DataFrame(unique_animals, columns=["id"])
df_genotype["genotype"] = None

In [17]:
# Save df_genotype as csv
df_genotype.to_csv("animal_genotypes_RawDataEdited.csv", index=False)